# Chapter 09 보충: 한 번에 실행 / LLM 검토 / 자습 과제 / 퀴즈

본문 실습(`ch09_regression.ipynb`)은 끝냈고, 여기서는 책 뒷부분을 이어서 정리함.
과제는 코드로 확인할 수 있는 건 직접 돌려서 확인했고, 글로 쓰는 과제는 문서로 적었음.

> 데이터는 가상으로 만든 거라 숫자 자체보다 **방법/순서** 가 중요한 실험들.

## 0. 준비

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.regression import (
    CATEGORICAL_FEATURES, FEATURE_COLUMNS, FORBIDDEN_FEATURES, NUMERIC_FEATURES, TARGET_COLUMN,
    build_regression_dataset, load_regression_source_data, make_one_hot_encoder,
    make_regression_models, public_prediction_result, run_regression_analysis,
    split_model_data_by_time,
)

processed_dir = project_root / "data" / "processed"
report_dir = project_root / "reports"

datasets = load_regression_source_data(processed_dir)
model_data = build_regression_dataset(datasets["customers"], datasets["orders"], datasets["order_items"])
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
X_train, y_train = train_data[FEATURE_COLUMNS], train_data[TARGET_COLUMN]
X_test, y_test = test_data[FEATURE_COLUMNS], test_data[TARGET_COLUMN]
print(model_data.shape, train_data.shape, test_data.shape)

(300, 11) (242, 11) (58, 11)


## 1. 한 번에 실행하기 (책 108번)

`run_regression_analysis()` 하나로 전체 파이프라인이 도는지 확인.
위 노트북이랑 같은 결과가 나와야 정상.

In [2]:
result = run_regression_analysis(
    processed_dir=processed_dir,
    report_dir=report_dir,
    test_size=0.2,
    random_state=42,
)
print("선택 모델:", result["selected_model_name"])
display(result["model_comparison"].round(2))
print("validation 전부 통과:", result["validation"]["passed"].all())

선택 모델: Random Forest


,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,95324.12,127790.68,156604.49,-0.00,0.00
1,Random Forest,selected_by_train_cv,74649.53,121476.77,160266.70,-0.05,4.94


validation 전부 통과: True


## 2. LLM 회귀 코드 검토 (책 109, 110번)

책에 나온 프롬프트를 LLM 에 그대로 물어보는 대신, 검증표 항목을
**내 코드(`src/regression.py`)에서 직접 확인**해 봤음. (LLM 답변은 이렇게 사람이 검증해야 한다는 게 요점)

In [3]:
source = (project_root / "src" / "regression.py").read_text(encoding="utf-8")

checks = []
def add(item, ok, note):
    checks.append({"검증 항목": item, "결과": "OK" if ok else "확인 필요", "확인 내용": note})

add("직접 누수", TARGET_COLUMN not in FEATURE_COLUMNS, "order_total 이 입력에 없음")
add("대리 누수", not ({"quantity", "unit_price", "line_total", "item_count", "total_quantity", "avg_unit_price"} & set(FEATURE_COLUMNS)),
    "수량/단가/상세 집계가 입력에 없음")
add("사후정보", "order_status" not in FEATURE_COLUMNS, "order_status 제외")
add("식별자", not ({"order_id", "customer_id", "product_id"} & set(FEATURE_COLUMNS)), "id 들 제외")
add("시간 분할", train_data["order_date"].max().normalize() < test_data["order_date"].min().normalize(),
    "train 마지막 날짜 < test 첫 날짜")
add("같은 날짜 겹침 없음", not (set(train_data["order_date"].dt.normalize()) & set(test_data["order_date"].dt.normalize())),
    "날짜 그룹 보존")
add("Pipeline 안 전처리", "fit_transform(" not in source, "src 코드에 Pipeline 밖 fit_transform 호출 없음")
add("선택은 train CV", "cross_validate(" in source and "select_diagnostic_model" in source,
    "cross_validate 결과로만 모델 선택")
add("Final Test는 두 모델만", set(result["model_comparison"]["model"]) == {"Baseline Mean", result["selected_model_name"]},
    "Baseline + 고정 모델")
add("지표 3개", {"test_MAE", "test_RMSE", "test_R2"} <= set(result["model_comparison"].columns), "MAE, RMSE, R2")
add("음수 R2 숨기지 않음", "clip" not in source and "max(0" not in source, "R2 를 0으로 자르는 코드 없음")
add("공개 결과 order_id 제거", "order_id" not in public_prediction_result(result["prediction_result"]).columns, "공개용에서 제거")

llm_check = pd.DataFrame(checks)
display(llm_check)

,검증 항목,결과,확인 내용
0,직접 누수,OK,order_total 이 입력에 없음
1,대리 누수,OK,수량/단가/상세 집계가 입력에 없음
2,사후정보,OK,order_status 제외
3,식별자,OK,id 들 제외
4,시간 분할,OK,train 마지막 날짜 < test 첫 날짜
5,같은 날짜 겹침 없음,OK,날짜 그룹 보존
6,Pipeline 안 전처리,OK,src 코드에 Pipeline 밖 fit_transform 호출 없음
7,선택은 train CV,OK,cross_validate 결과로만 모델 선택
8,Final Test는 두 모델만,OK,Baseline + 고정 모델
9,지표 3개,OK,"MAE, RMSE, R2"


## 3. 자습 과제

### 과제 1. 예측 시점 계약 작성 (주제: 고객의 다음 30일 구매 금액)

```
# 예측 문제 계약
## 업무 질문
이 고객이 앞으로 30일 동안 얼마나 살까?
## 예측 대상
고객의 다음 30일 구매 금액 합계
## 예측 단위
고객 한 명 (기준일마다 한 행)
## 예측 시점
매월 1일 아침 (기준일)
## 예측 시점에 알 수 있는 정보
기준일 이전 구매 횟수/금액, 가입 기간, 나이, 지역, 마지막 구매일
## 예측 시점에 알 수 없는 정보
기준일 이후 주문, 이후 취소/환불, 이후 쿠폰 사용
## 허용 입력 변수
최근 90일 구매 금액, 구매 횟수, 마지막 구매 후 경과일, 나이, 지역
## 금지 입력 변수
다음 30일 안의 주문 정보, 기준일 이후에 확정되는 주문 상태, customer_id
## 데이터 누수 위험
기준일 이후 주문이 집계에 섞이는 것. 환불로 나중에 바뀐 금액 쓰는 것.
## 평가 시점
과거 기간으로 학습하고 가장 최근 기간(미래)으로 평가
## 베이스라인
최근 90일 평균 구매 금액을 그대로 쓰는 방법, DummyRegressor(mean)
## 운영하지 않을 조건
베이스라인보다 MAE 가 개선되지 않거나 R2 가 음수일 때
```

### 과제 2. 누수 변수 분류

| 변수 | 허용 | 직접 누수 | 대리 누수 | 사후정보 | 식별자 | 이유 |
|---|---|---|---|---|---|---|
| order_total | X | O | | | | 목표값 자체 |
| quantity | X | | O | | | 목표값 계산 재료 |
| order_status | X | | | O | | 주문 처리 후에 정해질 수 있음 |
| customer_id | X | | | | O | 크기 의미 없는 id, 고객 암기 위험 |
| payment_method | O | | | | | 주문 메타데이터 |
| city | O | | | | | 고객 비식별 특성 |
| item_count | X | | O | | | 주문상세 집계라 금액이랑 연결됨 |
| order_dayofweek | O | | | | | 주문일에서 만든 값 |

### 과제 3. 누수 모델 실험 (학습용, 최종 결과로 안 씀)

안전한 입력 vs 거기에 `total_quantity`, `avg_unit_price` 를 더한 입력을 비교.

In [4]:
order_items = datasets["order_items"]
agg = order_items.groupby("order_id").agg(
    total_quantity=("quantity", "sum"),
    avg_unit_price=("unit_price", "mean"),
).reset_index()

train_leak = train_data.merge(agg, on="order_id", how="left", validate="one_to_one")
test_leak = test_data.merge(agg, on="order_id", how="left", validate="one_to_one")

LEAK_COLS = ["total_quantity", "avg_unit_price"]

def make_pipeline(model, numeric_cols):
    pre = ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", make_one_hot_encoder())]), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("preprocessor", pre), ("model", model)])

def evaluate(name, model_factory, numeric_cols):
    cols = numeric_cols + CATEGORICAL_FEATURES
    cv = TimeSeriesSplit(n_splits=5)
    cv_mae = -cross_val_score(make_pipeline(model_factory(), numeric_cols), train_leak[cols], y_train,
                              cv=cv, scoring="neg_mean_absolute_error").mean()
    pipe = make_pipeline(model_factory(), numeric_cols).fit(train_leak[cols], y_train)
    pred = pipe.predict(test_leak[cols])
    return {"입력": name, "cv_MAE": cv_mae, "test_MAE": mean_absolute_error(y_test, pred), "test_R2": r2_score(y_test, pred)}

rows = []
for label, cols in [("안전한 입력", NUMERIC_FEATURES), ("누수 입력 (+total_quantity, avg_unit_price)", NUMERIC_FEATURES + LEAK_COLS)]:
    r = evaluate(label, lambda: LinearRegression(), cols)
    r["모델"] = "Linear"
    rows.append(r)
    r = evaluate(label, lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1), cols)
    r["모델"] = "Random Forest"
    rows.append(r)

leak_result = pd.DataFrame(rows)[["모델", "입력", "cv_MAE", "test_MAE", "test_R2"]]
display(leak_result.round(2))

,모델,입력,cv_MAE,test_MAE,test_R2
0,Linear,안전한 입력,115127.62,123232.94,0.02
1,Random Forest,안전한 입력,99490.62,121476.77,-0.05
2,Linear,"누수 입력 (+total_quantity, avg_unit_price)",49875.02,42948.59,0.83
3,Random Forest,"누수 입력 (+total_quantity, avg_unit_price)",39571.59,26322.77,0.88


**정리**
- 누수 입력을 넣으면 MAE 가 확 줄고 R² 도 크게 올라감 (표 확인).
- 왜 좋아졌나: 수량 합계와 평균 단가는 주문금액을 거의 그대로 알려주는 재료라서.
- 실제 예측 시점엔 주문상세를 모르니까 쓸 수 없는 값이다. 점수가 높아도 **쓸 수 없는 모델** 이라 잘못된 모델.
- 그래서 이 실험 결과는 최종 결과로 채택 안 함.

### 과제 4. 무작위 분할 vs 시간 분할

In [5]:
def run_split(label, tr, te):
    models = make_regression_models(random_state=42)
    out = {"분할": label,
           "train 기간": f"{tr['order_date'].min().date()} ~ {tr['order_date'].max().date()}",
           "test 기간": f"{te['order_date'].min().date()} ~ {te['order_date'].max().date()}",
           "같은 날짜 겹침": len(set(tr["order_date"].dt.normalize()) & set(te["order_date"].dt.normalize()))}
    for name in ["Baseline Mean", "Random Forest"]:
        p = models[name].fit(tr[FEATURE_COLUMNS], tr[TARGET_COLUMN])
        pred = p.predict(te[FEATURE_COLUMNS])
        out[f"{name} MAE"] = mean_absolute_error(te[TARGET_COLUMN], pred)
        out[f"{name} R2"] = r2_score(te[TARGET_COLUMN], pred)
    return out

rand_train, rand_test = train_test_split(model_data, test_size=0.2, random_state=42)
split_compare = pd.DataFrame([
    run_split("무작위 분할", rand_train, rand_test),
    run_split("날짜 그룹 시간 분할", train_data, test_data),
])
display(split_compare.T)

,0,1
분할,무작위 분할,날짜 그룹 시간 분할
train 기간,2025-07-09 ~ 2026-07-08,2025-07-09 ~ 2026-04-08
test 기간,2025-07-14 ~ 2026-07-05,2026-04-10 ~ 2026-07-08
같은 날짜 겹침,27,0
Baseline Mean MAE,109144.027778,127790.681106
Baseline Mean R2,-0.001655,-0.00359
Random Forest MAE,112055.916632,121476.766989
Random Forest R2,-0.081771,-0.051076


**정리**
- 무작위 분할은 train/test 기간이 전부 겹치고 같은 날짜도 양쪽에 섞임 → 과거로 미래를 맞히는 상황이 아님.
- 실제 운영은 과거로 미래를 예측하니까 시간 분할 쪽이 더 가까운 평가.
- 결과: 무작위 분할은 같은 날짜가 27개나 양쪽에 겹쳤고, 이 분할에서는 Random Forest 가 Baseline 보다 MAE 가 오히려 높았음 (약 11.2만 vs 10.9만). 시간 분할에서는 Random Forest 가 약간 낮음 (약 12.1만 vs 12.8만).
- 즉 분할 방법만 바꿔도 결론이 달라질 수 있어서, 평가 구조를 실제 운영(과거 → 미래)과 맞추는 게 중요하다는 걸 확인. (가상 데이터라 숫자 자체로 어느 쪽이 맞다고 단정은 못 함)

### 과제 5. Pipeline 밖 전처리 실험 (잘못된 방식)

전체 데이터로 전처리 fit → 그 다음 train/test 분할. 이건 누수.

In [6]:
from sklearn.compose import ColumnTransformer as CT

# 잘못된 방식: 전체 X 로 fit
wrong_pre = CT([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), NUMERIC_FEATURES),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", make_one_hot_encoder())]), CATEGORICAL_FEATURES),
])
all_X = pd.concat([X_train, X_test])
wrong_pre.fit(all_X)                         # test 정보가 들어감
wrong_model = LinearRegression().fit(wrong_pre.transform(X_train), y_train)
wrong_pred = wrong_model.predict(wrong_pre.transform(X_test))

# 올바른 방식: Pipeline 안에서 train 만 fit
right = make_regression_models()["Linear Regression"].fit(X_train, y_train)
right_pred = right.predict(X_test)

pre_compare = pd.DataFrame([
    {"방식": "전체 데이터 fit 후 분할 (누수)", "test_MAE": mean_absolute_error(y_test, wrong_pred), "test_R2": r2_score(y_test, wrong_pred)},
    {"방식": "Pipeline (train만 fit)", "test_MAE": mean_absolute_error(y_test, right_pred), "test_R2": r2_score(y_test, right_pred)},
])
display(pre_compare.round(3))

# 스케일러 평균을 비교하면 test 정보가 섞였는지 눈으로 확인 가능
print("전체 fit 의 age 평균:", round(wrong_pre.named_transformers_["numeric"]["scaler"].mean_[2], 3))
print("train만 fit 의 age 평균:", round(right.named_steps["preprocessor"].named_transformers_["numeric"]["scaler"].mean_[2], 3))

,방식,test_MAE,test_R2
0,전체 데이터 fit 후 분할 (누수),123232.939,0.015
1,Pipeline (train만 fit),123232.939,0.015


전체 fit 의 age 평균: 42.297
train만 fit 의 age 평균: 42.488


**정리**
- 이번엔 두 방식의 MAE/R² 가 똑같이 나옴. 선형 회귀는 스케일러나 평균 채우기가 조금 달라져도 예측이 거의 안 변해서 그런 듯함.
- 그래도 age 평균이 42.297 (전체로 fit) vs 42.488 (train만 fit) 으로 다름 → 전체로 fit 하면 test 데이터 분포가 전처리에 들어간다는 증거.
- 점수에 티가 안 난다고 괜찮은 건 아니고, 모델/데이터에 따라 점수가 부풀 수 있어서 구조 자체를 막아야 함.

### 과제 6. CV 안정성 분석

In [7]:
cv_summary = result["cv_summary"]
display(cv_summary.round(2))
print("평균 MAE 가장 낮은 모델:", cv_summary.sort_values("cv_MAE_mean").iloc[0]["model"])
print("표준편차 가장 낮은 모델:", cv_summary.sort_values("cv_MAE_std").iloc[0]["model"])

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,95350.46,14097.40,-0.05
1,Linear Regression,5,115127.62,12535.45,-0.57
2,Random Forest,5,99490.62,11229.82,-0.22


평균 MAE 가장 낮은 모델: Baseline Mean
표준편차 가장 낮은 모델: Random Forest


- 평균 MAE 가 가장 낮은 건 Baseline Mean (95,350) 이고, 표준편차가 가장 낮은 건 Random Forest (11,230). Baseline 은 선택 대상이 아니라서, 선택 대상인 Linear vs Random Forest 만 보면 Random Forest 가 평균도 낮고 표준편차도 낮아서 선택 이유가 일관됨.
- 그런데 Random Forest 도 훈련 CV 에서는 Baseline 보다 MAE 가 높았음 (99,490 vs 95,350). 훈련 기간 안에서는 모델이 Baseline 을 못 이겼다는 뜻이라 Final Test 개선(4.9%)도 크게 믿기 어려움.
- 기간별로 성능이 달라지는 이유 후보: fold 마다 주문 수/기간이 다름, 극단적으로 큰 주문이 특정 기간에만 있음, 계절 영향 등. (원인 확정은 못 함)

### 과제 7. 테스트 봉인 규칙

```
## 테스트 사용 정책
- 테스트 데이터를 열 수 있는 시점: 입력 변수, 전처리, 후보, 선택 기준이 모두 고정된 뒤
- 테스트에서 계산할 지표: MAE, RMSE, R2, Baseline 대비 MAE 개선율
- 테스트를 보고 변경하면 안 되는 항목: 입력 변수, 하이퍼파라미터, 후보 모델, 선택 모델
- 변경이 필요할 때 확보할 새 데이터: 지금 test 보다 더 미래의 새 기간
- 최종 성능 보고 규칙: 고정한 모델과 Baseline 결과를 좋든 나쁘든 그대로 기록
```

### 과제 8. 오차 상위 사례 분석 (내부 예측 결과 상위 10건)

In [8]:
pred_all = result["prediction_result"]
top10 = pred_all.head(10)
display(top10.round(0))

print("과소예측 비율:", round((top10["residual"] > 0).mean() * 100), "%")
print("과대예측 비율:", round((top10["residual"] <= 0).mean() * 100), "%")
print("상위10 실제값 평균:", round(top10["actual_order_total"].mean()), " / 전체 test 실제값 평균:", round(pred_all["actual_order_total"].mean()))
print("상위10 날짜(월) 분포:")
print(pd.to_datetime(top10["order_date"]).dt.to_period("M").value_counts().sort_index())

,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,84,2026-04-13,730000.0,95883.0,634117.0,634117.0,Random Forest
1,264,2026-07-05,519000.0,137115.0,381885.0,381885.0,Random Forest
2,128,2026-04-22,414000.0,90436.0,323564.0,323564.0,Random Forest
3,183,2026-05-23,373000.0,102738.0,270262.0,270262.0,Random Forest
4,73,2026-06-14,382000.0,156907.0,225093.0,225093.0,Random Forest
5,51,2026-06-15,303000.0,87351.0,215649.0,215649.0,Random Forest
6,259,2026-05-11,312000.0,102117.0,209883.0,209883.0,Random Forest
7,34,2026-06-30,288000.0,83051.0,204949.0,204949.0,Random Forest
8,76,2026-06-14,367000.0,163519.0,203481.0,203481.0,Random Forest
9,96,2026-05-11,292000.0,94043.0,197957.0,197957.0,Random Forest


과소예측 비율: 100 %
과대예측 비율: 0 %
상위10 실제값 평균: 398000  / 전체 test 실제값 평균: 157552
상위10 날짜(월) 분포:
order_date
2026-04    2
2026-05    3
2026-06    4
2026-07    1
Freq: M, Name: count, dtype: int64


```
공통 패턴: 상위 오차 거의 전부 실제 금액이 큰 주문을 낮게 예측한 과소예측 (위 출력 확인)
과소예측 비율: 위 출력 참고 (대부분)
과대예측 비율: 위 출력 참고 (거의 없음)
매우 큰 실제값 존재: 있음 (최대 73만원)
특정 날짜 집중: 월별로 어느 정도 흩어져 있어서 한 날짜에 몰렸다고 보기 어려움
데이터 품질 문제 가능성: 이 데이터는 가상이라 확인 불가
현재 feature의 한계: 월/요일/나이/결제수단/성별/지역으로는 금액 크기를 구분 못 함
추가로 필요한 feature: 예측 시점 이전에 아는 정보 (예: 고객 과거 구매 금액, 장바구니 담은 상품 수 등)
원인으로 단정할 수 없는 내용: 이 주문들이 왜 컸는지는 모름. 점검 후보일 뿐.
```

### 과제 9. 베이스라인보다 못한 모델 보고서 (가정 상황)

> 실제 결과는 MAE 가 4.9% 개선됐지만 RMSE/R² 는 나빠서 애매했음. 아래는 "개선이 없다고 가정"했을 때 쓰는 양식.

```
## 결과
선택 모델의 test MAE 가 Baseline 보다 낮지 않다 (개선율 0 이하).
## 낮은 성능을 숨기지 않은 이유
숨기면 운영에서 잘못 믿고 쓸 수 있어서. 낮은 성능 자체가 중요한 결과.
## 누수 변수를 추가하지 않은 이유
quantity, total_quantity 같은 걸 넣으면 점수는 오르지만 예측 시점에 모르는 값이라 쓸 수 없음 (과제 3).
## 현재 입력 변수의 한계
월, 요일, 나이, 결제수단, 성별, 지역만으로는 주문금액 신호가 약함.
## 추가로 수집할 수 있는 예측 시점 이전 데이터
고객 과거 구매 이력, 최근 방문 기록, 장바구니 정보 (주문 전에 알 수 있는 것만).
## 새로운 검증 기간
지금 test 보다 더 이후의 새 기간 데이터로 다시 평가.
## 운영 보류 결정
Baseline 보다 나아지지 않았으므로 모델을 운영에 쓰지 않는다.
```

### 과제 10. 내부·공개 결과 분리

In [9]:
internal_df = pd.read_csv(report_dir / "ch09_regression_predictions_internal.csv")
public_df = public_prediction_result(internal_df)

assert "order_id" in internal_df.columns
assert "order_id" not in public_df.columns
print("내부 컬럼:", internal_df.columns.tolist())
print("공개 컬럼:", public_df.columns.tolist())
print("assert 통과")

내부 컬럼: ['order_id', 'order_date', 'actual_order_total', 'predicted_order_total', 'residual', 'abs_error', 'model']
공개 컬럼: ['order_date', 'actual_order_total', 'predicted_order_total', 'residual', 'abs_error', 'model']
assert 통과


## 4. 자기점검 퀴즈 (내 말로 답 적어봄)

1. **회귀 vs 분류?** 회귀는 금액처럼 연속 숫자를 예측, 분류는 완료/취소 같은 범주를 예측.
2. **수량·단가를 알면 회귀로 예측해야 하나?** 아니고 그냥 곱해서 계산하면 됨. 이번 건 주문상세를 모르는 시점을 가정한 교육용 문제.
3. **quantity 는 목표값이 아닌데 왜 누수?** unit_price 랑 곱하면 order_total 이 나오는 계산 재료라서 정답을 미리 알려주는 셈.
4. **order_status 를 빼는 이유?** 예측 시점 이후에 정해질 수 있는 사후정보라서.
5. **customer_id 숫자형이면 StandardScaler 써도 되나?** 안 됨. 크기 의미가 없는 식별자.
6. **MAE vs RMSE?** 둘 다 오차 크기. RMSE 가 큰 오차에 더 민감.
7. **R² 음수면 코드 오류?** 꼭 그렇진 않음. 평균 예측보다 못하다는 뜻일 수 있음 (이번 결과도 -0.05).
8. **왜 Dummy 랑 비교?** 단순 평균보다 진짜 나은지 알아야 해서.
9. **시간 데이터에서 무작위 분할이 위험한 이유?** 미래가 train 에 들어가서 평가가 쉬워짐.
10. **같은 날짜를 양쪽에 나누면 안 되는 이유?** 같은 날의 패턴이 양쪽에 섞여 평가가 쉬워짐.
11. **전처리를 Pipeline 안에 넣는 이유?** train fold 에서만 fit 되어서 test/validation 정보가 안 들어감.
12. **Final Test 에서 두 모델 다 평가하고 더 좋은 걸 골라도 되나?** 안 됨. 선택은 train CV 에서 끝내고 고정.
13. **선택 모델이 Dummy 보다 못하면 다른 후보로 바꿔도 되나?** 안 됨. 그대로 기록하고 새 기간으로 다시 검증.
14. **절대오차 큰 행은 원인이 확정된 건가?** 아님, 점검 후보.
15. **공개 보고서에 order_id 넣어도 되나?** 안 됨. 내부 파일에만.

## 5. Chapter 09 완료 체크리스트

이론
- [x] 회귀/분류 차이, 예측 시점, 누수 종류(직접·대리·사후정보), 식별자 문제
- [x] MAE/RMSE/R², Dummy 베이스라인, 시간 분할, 동일 날짜 경계, 테스트 역할
- [x] Pipeline 이 전처리 누수를 막는 이유, TimeSeriesSplit, 선택 고정 vs Final Test
- [x] 낮은 성능/음수 R² 의미, 내부·공개 결과 분리, LLM 코드 검증 기준

실습
- [x] processed 데이터 3개 확인, 목표값 생성, 1:1 / 다:1 관계 검증
- [x] Feature Audit, 누수 검사, 날짜 그룹 분할, 동일 날짜 중복 0
- [x] Pipeline, TimeSeriesSplit, 모델 선택 고정, Final Test(Dummy 비교)
- [x] Validation 저장, 내부 예측 / 공개 결과 분리, 그래프 2개, 체크리스트/보고서
- [x] preprocess_data.py, run_regression_analysis.py 실행
- [x] 낮은 성능 숨기지 않고 해석
- [ ] 책의 Chapter 05 전처리 파일은 없어서 가상 데이터로 대체 (한계)

## 다음: Chapter 10 분류 분석